# Notebook 11 – Date & Time Operations

Dates and times show up everywhere in real datasets — signup dates, order timestamps, log entries — but they rarely arrive in a clean, ready-to-use format. Before any time-based analysis (or a churn model that uses "days since last order" as a feature) can happen, raw date strings need to be parsed, converted, and enriched into usable features.

### The Dataset We'll Use Throughout

We continue with the same online retail customer story from Notebooks 7–10, now adding two date columns: `signup_date` (when the customer joined) and `last_order_date` (their most recent purchase).

- **Business angle:** Marketing wants to know how long customers have been with us, and which customers haven't ordered recently (a churn-risk signal).
- **AI/ML angle:** "Customer tenure in days" and "days since last order" are classic engineered features for a churn prediction model — both require solid date/time handling.

Let's rebuild the dataset with date columns, then work through each date/time technique.

In [1]:
import pandas as pd
import numpy as np
data = {
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name": ["Aarav", "Priya", "Rahul", "Sneha", "Vikram",
                       "Ananya", "Karthik", "Divya", "Manoj", "Lakshmi"],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze"],
    "total_orders":  [42, 15, 30, 5, 22, 60, 8, 35, 18, 3],
    "total_spend":   [125000, 32000, 98000, 8000, 45000,
                       210000, 12000, 87000, 39000, 4500],
    "signup_date":     ["2021-03-14", "2022-07-01", "2020-11-23", "2023-01-05", "2022-02-18",
                         "2019-08-30", "2023-04-10", "2020-06-15", "2022-09-27", "2023-03-01"],
    "last_order_date": ["2024-06-01", "2024-05-20", "2024-06-10", "2024-02-15", "2024-05-30",
                         "2024-06-08", "2024-01-20", "2024-06-05", "2024-04-18", "2023-12-01"]
}
df = pd.DataFrame(data)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,signup_date,last_order_date
0,101,Aarav,South,Gold,42,125000,2021-03-14,2024-06-01
1,102,Priya,North,Silver,15,32000,2022-07-01,2024-05-20
2,103,Rahul,South,Gold,30,98000,2020-11-23,2024-06-10
3,104,Sneha,West,Bronze,5,8000,2023-01-05,2024-02-15
4,105,Vikram,East,Silver,22,45000,2022-02-18,2024-05-30
5,106,Ananya,South,Gold,60,210000,2019-08-30,2024-06-08
6,107,Karthik,North,Bronze,8,12000,2023-04-10,2024-01-20
7,108,Divya,West,Gold,35,87000,2020-06-15,2024-06-05
8,109,Manoj,East,Silver,18,39000,2022-09-27,2024-04-18
9,110,Lakshmi,South,Bronze,3,4500,2023-03-01,2023-12-01


**Output Explanation:** Notice `signup_date` and `last_order_date` are currently stored as plain **strings** (`object` dtype) — pandas has no idea yet that these are dates. We'll fix that next.

In [2]:
df.dtypes

customer_id        int64
customer_name        str
region               str
membership           str
total_orders       int64
total_spend        int64
signup_date          str
last_order_date      str
dtype: object

**Output Explanation:** `signup_date` and `last_order_date` both show up as `object`, confirming they're just text right now. Every technique in this notebook builds toward converting them into a proper datetime type pandas can do arithmetic on.

## 1. Datetime — The Basic Building Block

### Concept Explanation
Python's built-in `datetime` module represents a single point in calendar time (year, month, day, hour, minute, second). Pandas builds on top of this with its own `Timestamp` type (covered next), but it's worth knowing the plain `datetime` object first, since pandas datetimes behave very similarly and interoperate with it directly.

### Business + AI/ML Example
Marketing wants to know **today's date** so they can compute how long each customer has been a member, and the churn pipeline needs a fixed "reference date" (the day the model is being scored) to compute tenure features consistently.

In [2]:
from datetime import datetime, date
now = datetime.now()
print("Current datetime:", now)
reference_date = datetime(2024, 6, 15)
print("Reference date:", reference_date)

Current datetime: 2026-08-20 13:09:32.139991
Reference date: 2024-06-15 00:00:00


**Output Explanation:** `datetime.now()` captures the exact moment the cell ran (down to microseconds), while `datetime(2024, 6, 15)` builds a specific fixed date. In a churn pipeline, using a fixed `reference_date` (rather than the ever-changing `now()`) keeps every scoring run comparable — everyone's tenure is measured "as of" the same day.

In [3]:
print("Year:", reference_date.year)
print("Month:", reference_date.month)
print("Day:", reference_date.day)
print("Is reference_date before now?", reference_date < now)

Year: 2024
Month: 6
Day: 15
Is reference_date before now? True


**Output Explanation:** A `datetime` object exposes `.year`, `.month`, `.day` directly, and supports normal comparison operators (`<`, `>`, `==`) since dates are naturally ordered. This is the foundation pandas' own datetime handling builds on.

## 2. `pd.Timestamp` — Pandas' Datetime Type

### Concept Explanation
`pd.Timestamp` is pandas' own datetime class — it's what you get when a column is converted to a proper date/time dtype. It behaves like Python's `datetime` but is optimized for use inside DataFrames/Series and supports pandas-specific features like time zones and nanosecond precision. When a whole *column* of `Timestamp` values is stored together, pandas gives it the dtype `datetime64[ns]`.

### Business + AI/ML Example
Before computing anything, the data science team wants a single, well-defined `Timestamp` representing "today" for a churn-scoring run, and wants to confirm it behaves consistently whether built from a string or from individual year/month/day values.

In [4]:
ts_from_string = pd.Timestamp("2024-06-15")
ts_from_parts = pd.Timestamp(year=2024, month=6, day=15)
print(ts_from_string)
print(ts_from_parts)
print("Are they equal?", ts_from_string == ts_from_parts)
print("Type:", type(ts_from_string))

2024-06-15 00:00:00
2024-06-15 00:00:00
Are they equal? True
Type: <class 'pandas.Timestamp'>


**Output Explanation:** Both construction methods — from a string and from explicit year/month/day arguments — produce the same `Timestamp`, confirming pandas parses the string unambiguously. `type()` shows this is a `pandas.Timestamp`, the building block behind every `datetime64[ns]` column.

In [5]:
sample_dates = pd.Series([pd.Timestamp("2024-01-01"), pd.Timestamp("2024-06-15")])
print(sample_dates)
print("dtype:", sample_dates.dtype)

0   2024-01-01
1   2024-06-15
dtype: datetime64[us]
dtype: datetime64[us]


**Output Explanation:** When multiple `Timestamp` values are packed into a `Series`, pandas stores them efficiently using the `datetime64[ns]` dtype — this is what enables fast vectorized date arithmetic across an entire column, instead of looping through Python `datetime` objects one at a time.

## 3. Date Parsing — Converting Strings to Datetimes

### Concept Explanation
`pd.to_datetime()` is the main tool for converting a column of date **strings** into real `datetime64[ns]` values. It can auto-detect common formats, accept an explicit `format=` string for speed and safety on messy data, and use `errors='coerce'` to turn unparseable values into `NaT` (pandas' "Not a Time" — the datetime equivalent of `NaN`) instead of crashing.

### Business + AI/ML Example
Our `signup_date` and `last_order_date` columns are still plain strings. Before any tenure or recency feature can be calculated, both must be converted to real datetimes — and the pipeline should be able to survive occasional badly-formatted dates without crashing.

In [6]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["last_order_date"] = pd.to_datetime(df["last_order_date"])
df.dtypes

customer_id                 int64
customer_name                 str
region                        str
membership                    str
total_orders                int64
total_spend                 int64
signup_date        datetime64[us]
last_order_date    datetime64[us]
dtype: object

**Output Explanation:** Both columns now show `datetime64[ns]` instead of `object` — the strings were successfully parsed. From this point on, pandas understands these columns as real dates and allows date arithmetic, comparisons, and the `.dt` accessor (used later for extracting features).

In [7]:
messy_dates = pd.Series(["2024-06-01", "not_a_date", "2024-07-15"])
parsed_safe = pd.to_datetime(messy_dates, errors="coerce")
parsed_safe

0   2024-06-01
1          NaT
2   2024-07-15
dtype: datetime64[us]

**Output Explanation:** `"not_a_date"` couldn't be parsed, so instead of raising an error and halting the whole pipeline, `errors='coerce'` replaced it with `NaT`. This is the safe default for real-world, occasionally messy data — bad rows can be flagged and handled later (e.g., dropped or investigated) rather than crashing the entire import.

## 4. Timedelta — Durations and Date Arithmetic

### Concept Explanation
`pd.Timedelta` represents a **duration** — a span of time, like "30 days" or "2 hours" — rather than a specific point in time. Subtracting two `Timestamp`s automatically produces a `Timedelta`, and you can also add/subtract a `Timedelta` to shift a date forward or backward. This is exactly how "days since X" features get built.

### Business + AI/ML Example
The churn model's single most important feature is **recency**: how many days have passed since each customer's last order, measured from our fixed reference date. Marketing also wants to know each customer's **tenure** (days since signup).

In [8]:
reference_date = pd.Timestamp("2024-06-15")
df["days_since_last_order"] = (reference_date - df["last_order_date"]).dt.days
df["tenure_days"] = (reference_date - df["signup_date"]).dt.days
df[["customer_name", "last_order_date", "days_since_last_order", "signup_date", "tenure_days"]]

,customer_name,last_order_date,days_since_last_order,signup_date,tenure_days
0,Aarav,2024-06-01,14,2021-03-14,1189
1,Priya,2024-05-20,26,2022-07-01,715
2,Rahul,2024-06-10,5,2020-11-23,1300
3,Sneha,2024-02-15,121,2023-01-05,527
4,Vikram,2024-05-30,16,2022-02-18,848
5,Ananya,2024-06-08,7,2019-08-30,1751
6,Karthik,2024-01-20,147,2023-04-10,432
7,Divya,2024-06-05,10,2020-06-15,1461
8,Manoj,2024-04-18,58,2022-09-27,627
9,Lakshmi,2023-12-01,197,2023-03-01,472


**Output Explanation:** Subtracting one `Timestamp` column from another produced a `Timedelta` for every row; `.dt.days` then extracted the whole number of days as a plain integer. Lakshmi (customer 110) has the highest `days_since_last_order`, flagging her as the customer who has gone longest without ordering — exactly the kind of recency feature a churn model relies on.

In [9]:
follow_up_deadline = df["last_order_date"] + pd.Timedelta(days=30)
df["follow_up_deadline"] = follow_up_deadline
df[["customer_name", "last_order_date", "follow_up_deadline"]].head(3)

,customer_name,last_order_date,follow_up_deadline
0,Aarav,2024-06-01,2024-07-01
1,Priya,2024-05-20,2024-06-19
2,Rahul,2024-06-10,2024-07-10


**Output Explanation:** Adding `pd.Timedelta(days=30)` to `last_order_date` computed a follow-up deadline exactly 30 days after each customer's last order — a common business rule ("re-engage a customer within 30 days of their last purchase") expressed in a single line of date arithmetic.

## 5. Time Zones

### Concept Explanation
By default, pandas `Timestamp`s are **timezone-naive** — they don't know which time zone they represent. `.dt.tz_localize()` attaches a time zone to naive datetimes (declaring "these times are in zone X"), and `.dt.tz_convert()` converts an already timezone-aware datetime to a *different* zone. This matters whenever data crosses regions — e.g., order timestamps logged in different local times, or a global business needing one consistent reporting time zone.

### Business + AI/ML Example
Our `last_order_date` values were recorded in **India Standard Time (IST)**, since that's where the business operates. Head office wants everything reported in **UTC** for consistency with other global systems.

In [10]:
last_order_ist = df["last_order_date"].dt.tz_localize("Asia/Kolkata")
print(last_order_ist.head(3))
print("dtype:", last_order_ist.dtype)

0   2024-06-01 00:00:00+05:30
1   2024-05-20 00:00:00+05:30
2   2024-06-10 00:00:00+05:30
Name: last_order_date, dtype: datetime64[us, Asia/Kolkata]
dtype: datetime64[us, Asia/Kolkata]


**Output Explanation:** `tz_localize("Asia/Kolkata")` attached the `+05:30` IST offset to each timestamp without changing the clock time itself — it's now explicitly marked as IST rather than an ambiguous, zone-less datetime. The dtype shows the attached time zone.

In [11]:
last_order_utc = last_order_ist.dt.tz_convert("UTC")
comparison = pd.DataFrame({
    "IST (original)": last_order_ist.head(3),
    "UTC (converted)": last_order_utc.head(3)
})
comparison

,IST (original),UTC (converted)
0,2024-06-01 00:00:00+05:30,2024-05-31 18:30:00+00:00
1,2024-05-20 00:00:00+05:30,2024-05-19 18:30:00+00:00
2,2024-06-10 00:00:00+05:30,2024-06-09 18:30:00+00:00


**Output Explanation:** `tz_convert("UTC")` shifted the displayed clock time back by 5 hours 30 minutes for each row — the same underlying instant in time, just expressed in a different time zone. This is the correct way to standardize timestamps from multiple regions before combining them in a single global report.

## 6. Extracting Date Features

### Concept Explanation
Once a column is a real `datetime64[ns]`, the `.dt` accessor unlocks a whole family of components: `.dt.year`, `.dt.month`, `.dt.day`, `.dt.dayofweek`, `.dt.quarter`, `.dt.day_name()`, and more. These become individual **features** — useful both for business breakdowns (e.g., "signups by month") and as direct inputs to a machine learning model, which can't use a raw datetime column but can absolutely use "signup month" or "is weekend" as numeric/categorical features.

### Business + AI/ML Example
Marketing wants to see which **month** most customers signed up in (for seasonal campaign planning), and the churn model needs `signup_year`, `signup_month`, and whether the last order fell on a **weekend** as explicit engineered features.

In [12]:
df["signup_year"] = df["signup_date"].dt.year
df["signup_month"] = df["signup_date"].dt.month
df["signup_quarter"] = df["signup_date"].dt.quarter
df["last_order_day_name"] = df["last_order_date"].dt.day_name()
df["last_order_is_weekend"] = df["last_order_date"].dt.dayofweek >= 5
df[["customer_name", "signup_date", "signup_year", "signup_month", "signup_quarter",
    "last_order_date", "last_order_day_name", "last_order_is_weekend"]]

,customer_name,signup_date,signup_year,signup_month,signup_quarter,last_order_date,last_order_day_name,last_order_is_weekend
0,Aarav,2021-03-14,2021,3,1,2024-06-01,Saturday,True
1,Priya,2022-07-01,2022,7,3,2024-05-20,Monday,False
2,Rahul,2020-11-23,2020,11,4,2024-06-10,Monday,False
3,Sneha,2023-01-05,2023,1,1,2024-02-15,Thursday,False
4,Vikram,2022-02-18,2022,2,1,2024-05-30,Thursday,False
5,Ananya,2019-08-30,2019,8,3,2024-06-08,Saturday,True
6,Karthik,2023-04-10,2023,4,2,2024-01-20,Saturday,True
7,Divya,2020-06-15,2020,6,2,2024-06-05,Wednesday,False
8,Manoj,2022-09-27,2022,9,3,2024-04-18,Thursday,False
9,Lakshmi,2023-03-01,2023,3,1,2023-12-01,Friday,False


**Output Explanation:** The `.dt` accessor pulled `signup_year`, `signup_month`, and `signup_quarter` straight out of `signup_date`, and `.dt.day_name()` turned `last_order_date` into a readable weekday name. `last_order_is_weekend` uses `.dt.dayofweek >= 5` (where Python's weekday numbering counts Saturday=5, Sunday=6) to flag weekend orders as a simple `True`/`False` feature — ready to feed directly into the churn model.

In [13]:
df.groupby("signup_quarter")["customer_id"].count().rename("num_signups")

signup_quarter
1    4
2    2
3    3
4    1
Name: num_signups, dtype: int64

**Output Explanation:** Grouping by the newly extracted `signup_quarter` gives marketing a quick signup-seasonality breakdown — this kind of aggregation is only possible once the date has been parsed and broken into components, which is exactly what this notebook built up step by step.